# Fine-tune the final Magi crop encoder block
Actual page → stored bbox → Magi crop/preprocessing → crop ViT → CLS token →
normalized embedding → cosine triplet loss. Cached embeddings never enter this path.

Only the last transformer block changes. Final LayerNorm and all earlier blocks
stay frozen. Unused OCR/detection modules and relation heads are omitted in memory;
original model files stay untouched. Run on a CUDA machine with the
existing Magi dependencies. Save only `magi_crop_last_block.pt` after selection.


In [6]:
import torch
import torch.nn.functional as F
import numpy as np
import hashlib, random
from pathlib import Path
from PIL import Image

ROOT = Path('C:/Users/ESHAAN/BYTE/Applied-Task/Magi/fineTuningMagiCharSimHeads')
DATA_FILE = ROOT / 'training_data.pt'
MODEL_FILE = ROOT / 'magi_crop_last_block.pt'
BASE_MODEL = Path('C:/MODELS/magiv2')

def file_hash(path):
    with Path(path).open('rb') as file:
        return hashlib.file_digest(file, 'sha256').hexdigest()

SEED, BATCH_SIZE, ENCODE_BATCH_SIZE = 42, 8, 32
EPOCHS, PATIENCE = 30, 5
LEARNING_RATE, WEIGHT_DECAY, MARGIN, BA_TOLERANCE = 1e-5, 1e-4, 0.2, 0.002
assert torch.cuda.is_available(), 'Run with CUDA-enabled PyTorch.'
device = torch.device('cuda:0')
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
data = torch.load(DATA_FILE, map_location='cpu', weights_only=True)
assert data['version'] == 2, 'Run the updated createTrainData.ipynb first.'
assert data['settings']['excluded_identities'] == ['NARRATION', 'UNKNOWN']
from collections import Counter
counts = Counter(data['appearances'][t['anchor_idx']]['sequence_idx'] for t in data['train_triplets'])
assert sum(counts.values()) > 0
for i in data['splits']['train']:
    print(data['sequences'][i]['id'], '| triplets:', counts[i],
          '| share:', f"{counts[i] / len(data['train_triplets']):.1%}")
assert max(counts.values()) / len(data['train_triplets']) <= data['settings']['max_sequence_share']
dataset_hash = file_hash(DATA_FILE)
base_hashes = {name: file_hash(BASE_MODEL / name) for name in ('config.json', 'pytorch_model.bin')}
print('GPU:', torch.cuda.get_device_name(device))


seq_b8b5563330fdc89d | triplets: 22 | share: 2.9%
seq_e8b4f75454f9491f | triplets: 14 | share: 1.8%
seq_5f9ee35846b7e635 | triplets: 8 | share: 1.0%
seq_b9b5a9ad9a64ced9 | triplets: 14 | share: 1.8%
seq_84541a1f4b159018 | triplets: 37 | share: 4.8%
seq_d749b853cf795b6a | triplets: 15 | share: 2.0%
seq_2032620aa4e4ac7f | triplets: 4 | share: 0.5%
seq_b136f32c2233b97c | triplets: 0 | share: 0.0%
seq_d8afceb0e8f9122d | triplets: 8 | share: 1.0%
seq_96839890d69d7698 | triplets: 6 | share: 0.8%
seq_45c9871f52e99417 | triplets: 22 | share: 2.9%
seq_f7f9d9dbf5b05cb5 | triplets: 29 | share: 3.8%
seq_da71f7a842c91090 | triplets: 21 | share: 2.7%
seq_952f154fb1505883 | triplets: 10 | share: 1.3%
seq_a3bd36201a0cd61b | triplets: 0 | share: 0.0%
seq_de542b63e6b19bc2 | triplets: 2 | share: 0.3%
seq_71033fec4de494fc | triplets: 28 | share: 3.7%
seq_d7af6c64ebc22bbe | triplets: 3 | share: 0.4%
seq_fa2eee6269a5dad3 | triplets: 22 | share: 2.9%
seq_06ec0acde37dffa4 | triplets: 7 | share: 0.9%
seq_f4928

## Inspect and freeze the model


In [7]:
from transformers import AutoConfig, AutoModel
from transformers.utils import logging as transformer_logging

# Skip unused OCR/detection modules; the original crop weights and processor stay identical.
config = AutoConfig.from_pretrained(str(BASE_MODEL), trust_remote_code=True, local_files_only=True)
config.disable_ocr = True
config.disable_detections = True
# The omitted modules produce a long expected warning; check loading details ourselves.
logger = transformer_logging.get_logger('transformers.modeling_utils')
old_level = logger.level
logger.setLevel('ERROR')
try:
    model, loading = AutoModel.from_pretrained(str(BASE_MODEL), config=config, trust_remote_code=True,
        local_files_only=True, output_loading_info=True)
finally:
    logger.setLevel(old_level)
assert not loading['missing_keys'] and not loading['mismatched_keys'] and not loading['error_msgs']
assert not any(name.startswith('crop_embedding_model.') for name in loading['unexpected_keys'])
encoder = model.crop_embedding_model
encoder.embeddings.config.mask_ratio = 0.0  # Magi disables MAE masking for crop embeddings.
assert encoder.config.hidden_size == 768
encoder.to(device)
model.eval()

print(encoder)
assert hasattr(encoder.encoder, 'layer') and len(encoder.encoder.layer) > 0
for parameter in model.parameters():
    parameter.requires_grad = False
last_block = encoder.encoder.layer[-1]
for parameter in last_block.parameters():
    parameter.requires_grad = True

prefix = f'crop_embedding_model.encoder.layer.{len(encoder.encoder.layer)-1}.'
trainable = {name: p for name, p in model.named_parameters() if p.requires_grad}
assert trainable and all(name.startswith(prefix) for name in trainable)
assert {id(p) for p in trainable.values()} == {id(p) for p in last_block.parameters()}
assert not any(p.requires_grad for p in encoder.layernorm.parameters())
for name, parameter in trainable.items():
    print(name, parameter.numel())
print('Trainable parameters:', sum(p.numel() for p in trainable.values()))
optimizer = torch.optim.AdamW(list(trainable.values()), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)


ViTMAEModel(
  (embeddings): ViTMAEEmbeddings(
    (patch_embeddings): ViTMAEPatchEmbeddings(
      (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
  )
  (encoder): ViTMAEEncoder(
    (layer): ModuleList(
      (0-11): 12 x ViTMAELayer(
        (attention): ViTMAEAttention(
          (attention): ViTMAESelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
          )
          (output): ViTMAESelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
        )
        (intermediate): ViTMAEIntermediate(
          (dense): Linear(in_features=768, out_features=3072, bias=True)
          (intermediate_act_fn): GELUActivation()
        )
        (output): ViTMAEOutput(
          (dense): Linear(i

## Crops and one forward pass per batch
The simple loader opens pages as needed. Validation encodes each appearance once
per epoch and holds those vectors in RAM. No full detection inference is run.


In [8]:
def load_crop(index):
    record = data['appearances'][index]
    with Image.open(record['page_path']) as image:
        page = np.asarray(image.convert('RGB'))
    return model.processor.crop_image(page, [record['bbox']])[0]

def encode_crops(crops):
    pixels = model.processor.preprocess_inputs_for_crop_embeddings(crops).to(device)
    output = encoder(pixels).last_hidden_state[:, 0]
    return F.normalize(output, dim=-1)

@torch.inference_mode()
def encode_appearances(indices):
    encoder.eval()
    embeddings = {}
    for start in range(0, len(indices), ENCODE_BATCH_SIZE):
        batch = indices[start:start + ENCODE_BATCH_SIZE]
        vectors = encode_crops([load_crop(index) for index in batch]).cpu()
        embeddings.update(zip(batch, vectors))
    return embeddings

from torch.utils.data import DataLoader

def collate_triplets(rows):
    # Keep anchors, positives, and negatives in that order for a single encoder call.
    return [load_crop(row[key]) for key in ('anchor_idx', 'positive_idx', 'negative_idx') for row in rows]

train_loader = DataLoader(data['train_triplets'], batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, collate_fn=collate_triplets)
val_indices = [index for i in data['splits']['validation'] for index in data['sequence_rows'][i]]

def cosine_triplet_loss(anchor, positive, negative):
    pos = F.cosine_similarity(anchor, positive, dim=-1)
    neg = F.cosine_similarity(anchor, negative, dim=-1)
    return (neg - pos + MARGIN).clamp(min=0).mean()

def train_epoch():
    model.eval()
    last_block.train()  # Enable training mode only for the unfrozen block.
    total, count = 0.0, 0
    for crops in train_loader:
        optimizer.zero_grad(set_to_none=True)
        z = encode_crops(crops)
        batch = len(crops) // 3
        loss = cosine_triplet_loss(z[:batch], z[batch:2*batch], z[2*batch:])
        assert torch.isfinite(loss)
        loss.backward()
        assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in trainable.values())
        assert all(p.grad is None for p in model.parameters() if not p.requires_grad)
        optimizer.step()
        total += loss.item() * batch
        count += batch
    return total / count


## Validation: threshold search uses only cross-page pairs

Choose the midpoint between distinct validation scores instead of an observed
score. Equal scores stay together. The test notebook reuses these saved thresholds.


In [9]:
def metrics(scores, labels, threshold):
    predicted = np.asarray(scores, dtype=np.float64) >= threshold
    positive, negative = labels == 1, labels == 0
    same = float(predicted[positive].mean()) if positive.any() else float('nan')
    different = float((~predicted[negative]).mean()) if negative.any() else float('nan')
    return dict(balanced_accuracy=(same + different) / 2,
                same_recall=same, different_recall=different,
                accuracy=float((predicted == labels).mean()) if len(labels) else float('nan'))

def choose_threshold(scores, labels):
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels)
    assert len(scores) == len(labels) and np.isfinite(scores).all()
    assert set(labels.tolist()) == {0, 1}
    order = np.argsort(-scores, kind='stable')
    values, targets = scores[order], labels[order]
    ends = np.r_[np.where(values[:-1] != values[1:])[0], len(values)-1]
    tp = np.cumsum(targets)[ends]
    fp = (ends + 1) - tp
    balanced = (tp / targets.sum() + 1 - fp / (len(targets)-targets.sum())) / 2
    best = int(np.argmax(balanced))
    if balanced[best] <= 0.5:
        # Use the empty SAME interval above the largest cosine score.
        return float((values[0] + 1.0) / 2) if values[0] < 1.0 else 1.0 + 1e-6
    upper = float(values[ends[best]])
    lower = float(values[ends[best] + 1]) if ends[best] + 1 < len(values) else -1.0
    return lower + (upper - lower) / 2  # Midpoint avoids placing the threshold on an observed score.


def validate():
    vectors = encode_appearances(val_indices)
    pairs = [pair for pair in data['val_pairs'] if pair['cross_page']]
    scores = np.array([float(vectors[p['a_idx']] @ vectors[p['b_idx']]) for p in pairs]).clip(-1, 1)
    labels = np.array([p['label'] for p in pairs])
    threshold = choose_threshold(scores, labels)
    result = metrics(scores, labels, threshold)
    rows = data['val_triplets']
    a = torch.stack([vectors[t['anchor_idx']] for t in rows])
    p = torch.stack([vectors[t['positive_idx']] for t in rows])
    n = torch.stack([vectors[t['negative_idx']] for t in rows])
    result.update(threshold=threshold, triplet_loss=float(cosine_triplet_loss(a, p, n)))
    assert all(np.isfinite(value) for value in result.values())
    return result

def block_state():
    return {name: parameter.detach().cpu().clone() for name, parameter in trainable.items()}

def report(epoch, train_loss, result):
    train_text = '-' if train_loss is None else f'{train_loss:.4f}'
    print(f"Epoch {epoch:02d} | train {train_text} | val {result['triplet_loss']:.4f} | "
          f"cross-BA {result['balanced_accuracy']:.4f} | same {result['same_recall']:.4f} | "
          f"diff {result['different_recall']:.4f} | threshold {result['threshold']:.4f}")


## Epoch 0, checkpoint selection, and early stopping
Original Magi is a candidate. BA improvements above 0.002 replace the candidate;
within that tolerance, lower validation triplet loss breaks ties. Compare against
the highest BA seen to prevent repeated small declines from drifting downward.
Only meaningful BA improvement resets patience; loss-only ties do not extend training.


In [10]:
baseline = validate()
best_epoch, best_result, best_state = 0, baseline.copy(), block_state()
peak_ba, patience_ba, bad_epochs = baseline['balanced_accuracy'], baseline['balanced_accuracy'], 0
history = [dict(epoch=0, train_loss=None, **baseline)]
report(0, None, baseline)

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch()
    result = validate()
    report(epoch, train_loss, result)
    history.append(dict(epoch=epoch, train_loss=train_loss, **result))
    ba = result['balanced_accuracy']
    peak_ba = max(peak_ba, ba)
    meaningful = ba > best_result['balanced_accuracy'] + BA_TOLERANCE
    tied = abs(ba - best_result['balanced_accuracy']) <= BA_TOLERANCE
    loss_better = result['triplet_loss'] < best_result['triplet_loss']
    if ba >= peak_ba - BA_TOLERANCE and (meaningful or (tied and loss_better) or
                                       best_result['balanced_accuracy'] < peak_ba - BA_TOLERANCE):
        best_epoch, best_result, best_state = epoch, result.copy(), block_state()
    if ba > patience_ba + BA_TOLERANCE:
        patience_ba, bad_epochs = ba, 0
    else:
        bad_epochs += 1
    if bad_epochs >= PATIENCE:
        print('Early stopping: no meaningful cross-page BA improvement.')
        break

assert file_hash(DATA_FILE) == dataset_hash, 'Dataset changed during training.'
saved = dict(version=2, state_dict=best_state, trainable_prefix=prefix, best_epoch=best_epoch,
    settings=dict(seed=SEED, learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
        margin=MARGIN, trained_blocks=1, ba_tolerance=BA_TOLERANCE, patience=PATIENCE,
        batch_size=BATCH_SIZE, mask_ratio=0.0, threshold_selection='midpoint'),
    validation=best_result, baseline_validation=baseline, dataset_hash=dataset_hash,
    base_model_hashes=base_hashes)
torch.save(saved, MODEL_FILE)
parameters = dict(model.named_parameters())
with torch.no_grad():
    for name, tensor in best_state.items():
        parameters[name].copy_(tensor.to(device))
model.eval()
print('Best epoch:', best_epoch, '| cross-BA:', round(best_result['balanced_accuracy'], 4))
print('Saved:', MODEL_FILE)


Epoch 00 | train - | val 0.1289 | cross-BA 0.7667 | same 0.7018 | diff 0.8316 | threshold 0.6727
Epoch 01 | train 0.1217 | val 0.1274 | cross-BA 0.7684 | same 0.7018 | diff 0.8351 | threshold 0.6879
Epoch 02 | train 0.1184 | val 0.1261 | cross-BA 0.7694 | same 0.7105 | diff 0.8282 | threshold 0.6898
Epoch 03 | train 0.1153 | val 0.1250 | cross-BA 0.7694 | same 0.7105 | diff 0.8282 | threshold 0.6966
Epoch 04 | train 0.1126 | val 0.1242 | cross-BA 0.7745 | same 0.7105 | diff 0.8385 | threshold 0.7035
Epoch 05 | train 0.1098 | val 0.1234 | cross-BA 0.7734 | same 0.6842 | diff 0.8625 | threshold 0.7332
Epoch 06 | train 0.1072 | val 0.1228 | cross-BA 0.7770 | same 0.7018 | diff 0.8522 | threshold 0.7221
Epoch 07 | train 0.1047 | val 0.1223 | cross-BA 0.7770 | same 0.7018 | diff 0.8522 | threshold 0.7240
Epoch 08 | train 0.1022 | val 0.1216 | cross-BA 0.7804 | same 0.7018 | diff 0.8591 | threshold 0.7301
Epoch 09 | train 0.0996 | val 0.1214 | cross-BA 0.7821 | same 0.7018 | diff 0.8625 | th